# Chain

**Chain**(체인)은 여러 컴포넌트(요소)를 정해진 순서대로 연결하여 **복잡한 AI 작업을 단계별로 자동화**할 수 있도록 돕는 구조이다.

- 각 컴포넌트는 **이전 처리결과를 입력으로 받아 처리한 후 다음 단계로 결과를 전달**한다.
- 복잡한 작업을 여러 개의 단순한 단계로 나누고, 각 단계를 순차적으로 실행함으로써 전체 작업을 체계적으로 구성할 수 있다.

## 기본 개념

- 체인은 하나의 LLM 호출에 그치지 않고 **여러 LLM 호출이나 도구 실행등을 순차적으로 연결**하여 실행 할 수 있다.
- 예를 들어, 사용자의 질문 → 검색 → 요약 → 응답 생성 같은 일련의 작업을 체인으로 구성할 수 있다.
- 이러한 체인구조를 사용하면 **작업흐름이 명확**해지고 **코드의 재사용성**이 높아지며 **유지 보수 및 확장성이** 향상된다.

## LangChain에서의 Chain 구성 방식

LangChain은 다음 두 가지 방식을 통해 체인을 구성할 수 있다.

### 1. Off-the-shelf Chains 방식 (클래식 방식)

- LangChain에서 제공하는 **미리 정의된 Chain 클래스**(예: `LLMChain`, `SequentialChain`, `SimpleSequentialChain`)를 활용하는 방식이다.
- 각 클래스는 다양한 chain 알고리즘들을 미리 구현한 것으로 상황에 맞는 것을 선택하여 필요한 구성요소를 전달해 생생한다.
- 이 방식은 LangChain의 **초기 방식**이며, 새로운 기능 확장이나 유연한 구성에 한계가 있기 때문에 현재 **더 이상 사용되지 않음(deprecated)** 상태이다.
  - 현재 LangChain에서는 권장하지 않는 방식이다.

### 2. LCEL (LangChain Expression Language) 방식

- LCEL은 체인을 표현식(Expression) 기반의 선언적 파이프라인 방식으로 구성할 수 있도록 설계된 최신 체인 구성 방법이다. 
- 각 컴포넌트들을 `|` 연산자로 연결하여, 흐름이 자연스럽게 이어지는 형태의 체인을 구성한다.
- LCEL 방식은 간결하고 선언적인 문법을 제공하여 **직관적이고 융통성과 확장성 있는 체인 구성**이 가능하다.
- LCEL은
  - 선형적 흐름 구조를 가진다.
  - 문법이 간결하고 선언적이다.
  - 체인의 구조가 코드만 봐도 쉽게 파악된다.
  - 유연하고 확장성이 매우 뛰어나다.
- `Runnable` 기반 구조
  - LCEL방식을 구성하는 모든 컴포넌트들은 `Runnable` 이라는 공통 인터페이스를 기반으로 동작한다.
  - 체인을 구성하는 각 컴포넌트들은 `Runnable` 을 상속하여 구현하여 이를 통해 일관된 실행 인터페이스를 제공한다.
  - **공통 메소드**:
    - `invoke()`: 단일 입력에 대한 처리
    - `batch()`: 다수 입력을 묶어서 한번에 처리
    - `stream()`: 스트리밍 방식의 요청
    - `ainvoke()`, `abatch()`, `astream()`: 비동기적 처리 메소드

In [6]:
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
prompt = ChatPromptTemplate.from_template(
    template="{item}에 어울리는 브랜드 이름 {count}개를 만들어 주세요."
)
model = ChatOpenAI(model="gpt-5.4-nano")
parser = StrOutputParser()

In [7]:
query = prompt.invoke({"item":"가방", "count":5})
res = model.invoke(query)
result = parser.invoke(res)
print(result)

1. 어울림백（Eo-Uri Rim Bag）  
2. 다정가방（Dajeong Bag）  
3. 걸음선（Georeumseon）  
4. 노을바이브백（Noeul Vibe Bag）  
5. 가벼운하루（Gabyeoun Haru）


In [2]:
# !uv pip install langchain-classic

In [8]:
############################################################
# 기존의 Off the shell 방식 - langchain-classic 설치 필요
############################################################
from langchain_classic import LLMChain
# chain을 구성하는 요소들을 넣어서 생성.
# prompt_template-[prompt]->model-[응답]->output parser-> 최종결과
chain = LLMChain(
    prompt=prompt,
    llm=model,
    output_parser=parser
)

res = chain.invoke({"item":"가방", "count":3})
print(res)


C:\Users\Playdata\AppData\Local\Temp\ipykernel_22224\1453150728.py:7: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(


{'item': '가방', 'count': 3, 'text': '1. 루미아오라  \n2. 노빌라피오  \n3. 트래블노트'}


In [10]:
##############################
#  LCEL
##############################
chain2 = prompt | model | parser
print(type(chain2))
res2 = chain2.invoke({"item":"TV", "count":3}) 

<class 'langchain_core.runnables.base.RunnableSequence'>


In [12]:
print(res2)

아래는 TV(텔레비전) 브랜드로 어울릴 만한 **브랜드 이름 3개**입니다.

1. **선명TV**  
2. **빛가람**  
3. **울림뷰**


In [15]:
from langchain_core.runnables import Runnable
print(isinstance(model, Runnable), isinstance(prompt, Runnable), 
      isinstance(parser, Runnable), isinstance(chain2, Runnable))

True True True True


# Runnable 타입 주요 클래스


## [Runnable](https://reference.langchain.com/python/langchain_core/runnables/#langchain_core.runnables.base.Runnable)
- LangChain의 Runnable은 실행 가능한 작업 단위를 캡슐화한 개념으로, 데이터 흐름의 각 단계를 정의하고 **체인(chain) 에 포함 되어**  복잡한 작업의 각 단계를 수행 한다.
- **Chain을 구성하는 class들**은 Runnable의 상속 받아 구현한다.
- **Prompt Template클래스**, **Chat 모델**, **Output Parser 클래스** 등 다양한 컴포넌트가 Runnable을 상속받아 구현된다.

### 주요 특징
- 작업 단위의 캡슐화:
    - Runnable은 특정 작업(예: 프롬프트 생성, LLM 호출, 출력 파싱 등)을 수행하는 독립적인 컴포넌트이다.
    - 각 컴포넌트는 독립적으로 테스트 및 재사용이 가능하며, 조합하여 복잡한 체인을 구성할 수 있다.
- 체인 연결 및 작업 흐름 관리:
    - Runnable은 체인(chain, 일련의 연결된 작업 흐름)을 구성하는 기본 단위로 사용된다.
    - LangChain Expression Language(LCEL)를 사용하면 | 연산자를 통해 여러 Runnable을 쉽게 연결할 수 있다.
    - 입력과 출력의 형식을 일관되게 유지하여 각 단계가 자연스럽게 연결된다.
- 모듈화 및 디버깅 용이성:
    - 각 단계가 명확히 분리되어 문제 발생 시 어느 단계에서 오류가 발생했는지 쉽게 확인할 수 있다.
    - 복잡한 작업을 작은 단위로 나누어 체계적으로 관리할 수 있다.
      
### Runnable의 표준 메소드
- 모든 Runnable이 구현하는 공통 메소드
    - **`invoke(input, config:RunnableConfig)->output`**: 단일 입력을 처리하여 결과를 반환.
    - **`batch(input:list, config:RunnableConfig|list[RunnableConfig]) -> list[Output]`**: 여러 입력 데이터들을 한 번에 처리.
    - **`stream(input, config:RunnableConfig) -> Iterator[Output]`**: 입력에 대해 스트리밍 방식으로 응답을 반환.
    - **`assign(**kwargs)`**:
      -  앞 Runnable의 출력 결과에 새로운 key–value 쌍의 Field 추가(assign) 하여 다음 Runnable로 전달.
      -  값으로는 Runnable 객체(LCEL체인등)나 고정 값(리터럴) 모두 가능하며, 각 항목은 실행 시 평가되어 기존 출력에 병합한다.
      -  주로 앞 단계의 출력에 부가 정보(field)를 추가하고자 할 때 사용한다. 특히 `RunnablePassthrough`와 결합해, 입력을 그대로 넘기면서 특정 field만 추가할 때 자주 사용


### Runnable의 주요 구현체(하위 클래스)

- 다음 클래스들은 기능을 제공하는 것이 아니라 **chain 구조를 다양하게 구성** 할 수 있도록 도와주는 **Runnable** 타입의 클래스들이다.

- **`RunnableSequence`**
    - 여러 `Runnable`을 순차적으로 연결하여 실행하는 구성이다.
    - 각 단계의 출력이 다음 단계의 입력으로 전달된다.
    - 보통은 LCEL 문법을 사용해서 정의한다.
      - LCEL을 사용하여 체인을 구성할 경우 자동으로 `RunnableSequence`로 변환된다.


In [18]:
from langchain_core.runnables import RunnableSequence

# chain = RunnableSequence(prompt, model, parser)
chain = prompt | model | parser
# chain
chain.invoke({"item":"물", "count":2})

'1. **물빛하우스**  \n2. **청아수림**'


- **`RunnableLambda`**
    - Lambda 표현식의 함수를 `Runnable`로 변환할 때 사용한다.    
    - 일반함수도 `RunnableLambda`로 변환할 수 있다. 단 일반 함수는 변환 없이 chain에 포함 시킬 수 있기 때문에 굳이 변환할 필요가 없다.
    - Runnable로 만들 함수 구문
        - parameter: 입력 값 1개선언.
        - return: 다음 chain에 전달할 값의 형식


In [21]:
from langchain_core.runnables import RunnableLambda

# RunnableLambda(함수) 
# 함수-파라미터(1개 -> 앞 체인으로 부터 받을 값에 맞춘다.)
#     - 리턴값 -> 다음 체인의 입력 타입에 맞게 반환.
c1 = RunnableLambda(lambda input_data:f"{input_data}에 대해서 한 문장으로 설명해줘.")
# type(c1)
c1.invoke("LLM 모델")

'LLM 모델에 대해서 한 문장으로 설명해줘.'

In [22]:
chain = c1 | model | parser
chain.invoke("LLM모델 ") #chain 호출시에는 첫번째 컴포넌트에 전달할 값을 넣어서 호출

'LLM(대규모 언어 모델)은 방대한 텍스트를 학습해 문맥을 이해하고 자연어를 생성·예측하는 인공지능 모델입니다.'

In [25]:
prompt = ChatPromptTemplate(
    messages = [
        ("system", "모든 응답은 100글자 이내로 작성해줘."),
        ("user", "{query}")
    ]
)
model = ChatOpenAI(model="gpt-5.4-mini")
parser = StrOutputParser()

# Chain구성. chain 응답: LLM 응답내용, 글자수
chain = prompt | model | parser | RunnableLambda(lambda x : (x, len(x)))

In [26]:
res = chain.invoke("AI에 대해서 설명해줘.")
print(res)

('AI는 사람처럼 학습·추론하는 컴퓨터 기술입니다. 데이터로 패턴을 배우고 예측합니다.', 47)


In [ ]:
def get_value_len(value:str):
    return value, len(value)

# 일반함수(callable)를 chain의 구성으로 포함시킬 수 있다. -> 내부적으로 Runnable로 변환되서 들어간다.
chain2 = prompt | model | parser | get_value_len   # RunnableLambda(get_value_len) 
chain2.invoke("크리스마스")

('메리 크리스마스! 🎄', 11)


-  **`RunnablePassthrough`**
    - 입력 데이터를 가공하지 않고 그대로 다음 단계로 전달하는 `Runnable`이다.
      - 앞 Runnable으로 부터 전달 받은 **입력 값을 다음 Runnable로 그대로 전달**한다.
           - `RunnablePassthrough()`
      - 입력받은 값에 **Field를 추가**해서 전달할 경우 `assign()` 메소드를 사용한다.
           - `RunnablePassthrough.assign(new_key1="new_value1", new_key2="new_value2", ..)`


In [31]:
from langchain_core.runnables import RunnablePassthrough

rp = RunnablePassthrough() # 단순히 받은 값을 다음으로 통과시킨다.

result = rp.invoke("안녕하세요")
result = rp.invoke([1, 2, 3, 4, 5])
result = rp.invoke({"a":10, "b":20})
print(result)

{'a': 10, 'b': 20}


In [ ]:
# 입력받은 값(딕셔너리)에 item을 추가해서 다음으로 전달.
r1 = RunnableLambda(lambda x: "서울시 금천구 독산동")
r2 = RunnableLambda(lambda x: "010-1111-2222")

# assign(key=Runnable, ...)
## 입력받은 딕셔너리에 address와 tel_no key를 추가. value는 Runnable을 호출해서 반환된 값을 설정.
rp2 = RunnablePassthrough.assign(
    address=r1,
    tel_no=r2
)
result = rp2.invoke({"name":"홍길동"})
result

{'name': '홍길동', 'address': '서울시 금천구 독산동', 'tel_no': '010-1111-2222'}


- **`RunnableParallel`**
    - 여러 `Runnable`을 병렬로 실행한 후, 결과를 결합하여 다음 단계로 전달한다.
    - 
        ```python
        RunnableParallel(
            {
                "key1":Runnable1, 
                "key2":Runnable2,
                "key3":Runnable3, ...
            }
        )
        ```
    - 각 Runnable의 실행결과를 Value로 Dictionary를 생성해서 반환한다.
    - LCEL로 정의할 때는 Chain에 dictionary로 정의한다.



In [ ]:
from langchain_core.runnables import RunnableParallel

r1 = RunnableLambda(lambda x : x + 10)
r2 = RunnableLambda(lambda x : x - 10)
r3 = RunnableLambda(lambda x : x * 10)
r4 = RunnableLambda(lambda x : x / 10)

# c = r1 | r2 | r3 | r4 
parallel = RunnableParallel(
    {
        "value1": r1,
        "value2": r2,
        "value3": r3,
        "value4": r4,
        "org_value": RunnablePassthrough() # 입력받은 값을 그대로 다음으로 넘겨야 할 경우.
    }
)

result = parallel.invoke(200)
result

{'value1': 210,
 'value2': 190,
 'value3': 2000,
 'value4': 20.0,
 'org_value': 200}

In [35]:
c = RunnablePassthrough() | {
        "value1": r1,
        "value2": r2,
        "value3": r3,
        "value4": r4,
        "org_value": RunnablePassthrough() # 입력받은 값을 그대로 다음으로 넘겨야 할 경우.
    }
c.invoke(2000)

{'value1': 2010,
 'value2': 1990,
 'value3': 20000,
 'value4': 200.0,
 'org_value': 2000}

### LCEL Chain 예제

In [ ]:
##################################################
# TODO 1
# 음식 이름을 입력하면 그 음식의 레시피를 llm이 출력하는 Chain을 LCEL 을 이용해서 구성한다.
# 입력 : 음식 이름 - recipe_chain.invoke({"food":"김치찌게"})
# 출력 : 음식의 레시피 - 김치찌게 레시피. 

# chain구성: prompt_template -> model(gpt-5-mini) -> StrOutputParser

In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

system_prompt = """
<instruction>
당신은 요리전문 AI Assistant입니다.
요청받은 음식의 레시피를 자세하고 쉽게 작성해 주세요.
출력 방법은 아래 output_format을 참고해서 응답해주세요.
</instruction>

<output_format>
- Markdown 형식으로 답변을 작성합니다.
- 응답 내용에는 다음 항목들을 포함합니다.
    - 요리 이름
    - 요리 기본정보
        - 난이도
        - 조리시간
        - 인분
    - 요리에 필요한 재료
    - 요리 방법
    - 팁
</output_format>
"""
parser = StrOutputParser()
model = ChatOpenAI(model="gpt-5.4-mini")
prompt = ChatPromptTemplate(
    messages = [
        ("system", system_prompt),
        ("user", "{food}의 레시피를 작성해 주세요.")
    ]
)

In [2]:
recipe_chain = prompt | model | parser

res = recipe_chain.invoke({"food":"김치찌게"})

In [5]:
from IPython.display import Markdown
print(res)
# Markdown(res)

# 김치찌개 레시피

## 요리 기본정보
- **난이도:** 쉬움
- **조리시간:** 약 25~30분
- **인분:** 2~3인분

## 요리에 필요한 재료
- 잘 익은 김치 1컵~1.5컵
- 돼지고기(목살 또는 앞다리살) 150g  
  - 없으면 참치 1캔 또는 두부로 대체 가능
- 두부 1/2모
- 양파 1/2개
- 대파 1대
- 다진 마늘 1큰술
- 고춧가루 1큰술
- 국간장 1큰술
- 설탕 1작은술
- 김치 국물 1/2컵
- 물 또는 육수 2.5~3컵
- 참기름 1작은술
- 식용유 1작은술
- 소금 약간
- 후추 약간

## 요리 방법
1. **재료 준비하기**  
   김치는 먹기 좋게 썰고, 돼지고기는 한입 크기로 썰어 준비합니다.  
   두부는 두툼하게 썰고, 양파는 채 썰고, 대파는 어슷썰기 합니다.

2. **고기와 김치 볶기**  
   냄비에 식용유와 참기름을 넣고 중불로 달군 뒤 돼지고기를 먼저 볶습니다.  
   고기가 어느 정도 익으면 김치를 넣고 2~3분 정도 함께 볶아 김치의 신맛과 감칠맛을 살려줍니다.

3. **양념 넣기**  
   고춧가루, 다진 마늘, 국간장, 설탕을 넣고 골고루 섞어 볶습니다.  
   김치 국물도 함께 넣어 맛을 더 진하게 만듭니다.

4. **끓이기**  
   물 또는 육수를 부은 뒤 센 불에서 끓입니다.  
   끓기 시작하면 중불로 줄여 10~15분 정도 푹 끓여줍니다.

5. **마무리 재료 넣기**  
   양파와 두부를 넣고 3~5분 더 끓입니다.  
   마지막에 대파를 넣고, 부족한 간은 소금이나 국간장으로 맞춥니다.  
   후추를 약간 뿌려 마무리합니다.

## 팁
- **김치는 잘 익은 신김치**를 사용하면 국물 맛이 훨씬 깊어집니다.
- 돼지고기는 **목살이나 앞다리살**을 사용하면 국물이 더 진하고 맛있습니다.
- 국물이 더 진한 맛을 원하면 **멸치육수나 다시마육수**를 사용해 보세요.
- 참치김치찌개로 만들 경우, 돼지고기 대신 **참치 1캔**을 마지막 단계에 넣으면 

In [6]:
res2 = recipe_chain.invoke({"food":"봉골레 파스타"})

In [7]:
print(res2)

# 봉골레 파스타 레시피

## 요리 기본정보
- **난이도:** 중
- **조리시간:** 약 20~25분
- **인분:** 2인분

## 요리에 필요한 재료
### 주재료
- 스파게티 면 160g
- 바지락 300g
- 마늘 6~8쪽
- 페페론치노 4~6개
- 올리브오일 4큰술
- 화이트와인 1/2컵(선택)
- 소금 약간
- 후추 약간
- 이탈리안 파슬리 약간

### 선택 재료
- 버터 1큰술
- 면수 1/2컵 정도
- 레몬즙 약간
- 파마산 치즈 약간(전통적인 봉골레에는 생략 가능)

## 요리 방법
1. **바지락 해감하기**
   - 바지락은 소금물에 담가 1~2시간 정도 해감한 뒤 깨끗이 씻어 주세요.
   - 껍데기끼리 비벼 씻으면 이물질 제거에 도움이 됩니다.

2. **재료 손질하기**
   - 마늘은 편으로 썰거나 얇게 다져 주세요.
   - 페페론치노는 부셔서 매운맛이 잘 우러나도록 준비합니다.
   - 파슬리는 잘게 다져 둡니다.

3. **면 삶기**
   - 끓는 물에 소금을 넣고 스파게티 면을 포장지보다 1~2분 짧게 삶아 주세요.
   - 면수는 나중에 소스 농도를 맞추는 데 사용하므로 조금 남겨 둡니다.

4. **봉골레 소스 만들기**
   - 팬에 올리브오일을 두르고 약불에서 마늘과 페페론치노를 넣어 천천히 볶아 향을 냅니다.
   - 마늘이 노릇해지기 시작하면 바지락을 넣고 중불로 올립니다.
   - 화이트와인을 넣고 뚜껑을 덮어 바지락이 입을 벌릴 때까지 익혀 주세요.

5. **면과 소스 섞기**
   - 바지락이 대부분 열리면 삶아 둔 면을 넣고 잘 섞어 줍니다.
   - 면수가 부족하면 조금씩 넣어가며 소스가 면에 잘 감기도록 볶습니다.
   - 간을 보고 소금, 후추로 맞춰 주세요.

6. **마무리**
   - 불을 끈 뒤 파슬리와 버터를 넣어 한 번 더 섞으면 풍미가 좋아집니다.
   - 취향에 따라 레몬즙을 살짝 넣어 상큼하게 마무리해도 좋습니다.

## 팁
- **바지락 해감이 중요**합니다. 해감이 덜 되면

In [ ]:
##############################################################
#  TODO 2
# 번역할 내용, 번역할 언어 를 입력하면 내용을 그 언어로 번역하는 Chain을 LCEL 을 이용해서 구성한다.
#
## 입력: 번역할 내용, 언어.  translate_chain.invoke({"content":"안녕하세요.", "language":"영어"})
## 출력: "번역할 내용"을 "언어" 로 번역한 결과 - "How are you?".

# chain구성: prompt_template -> model(gpt-5-mini) -> StrOutputParser

### Chain과 Chain간의 연결

## 함수를 Runnable로 정의하기

### 함수 구현
- **파라미터**
   - 이전 Chain에서 출력한 값을 입력으로 받을 수 있도록 정의한다.
- **리턴값**
   - 다음 Chain으로 입력할 값을 반환하도록 구현한다.

### Runnable 타입으로 만들기
1. LCEL Chain안에 함수를 구성요소로 포함시키면, 그 함수는 자동으로 `Runnable` 로 취급된다.
   - 별도의 래핑이나 추가 처리는 필요하지 않다.
2. `RunnableLambda()` 에 넣어 명시적으로 `Runnable` 타입으로 만든다.
   - Lambda 표현식으로 정의할 경우 `RunnableLambda(lambda 표현식)` 으로 정의해야 한다.
   - 보통 일반함수는 `RunnableLambda`를 사용할 필요 없다.
3. `@chain` decorator를 사용
   - 함수에 `@chain` decorator가 선언되면 그 함수는 `RunnableLambda` 타입이 된다.
   - 이 방식은 LCEL만으로 표현하기 어려운 실행 흐름을 직접 정의해야 할 때 주로 사용된다.
     - LCEL은 순차 실행구조를 따른다. 그래서  제어문을 이용해 그 흐름을 제어할 수가 없다. 
     - 단순한 파이프라인에서는 LCEL만으로도 충분하지만, 다음과 같은 경우에는 한계가 있다.
       - 특정 단계를 조건에 따라 실행하거나 생략해야 하는 경우
       - 동일한 단계를 반복적으로 실행해야 하는 경우
       - 여러 판단 로직에 따라 실행 경로가 달라지는 agent 구조
     - 이처럼 복잡한 업무 흐름을 가지는 agent는 단순한 순차 구조만으로는 원하는 응답 품질을 얻기 어렵다. 결국 실행 흐름 자체를 개발자가 직접 코드로 정의해야 하며, 이러한 경우에 `@chain`을 사용해 chain/agent 함수를 구현한다.
   - 이러한 복잡한 실행 흐름을 보다 구조적으로 정의하기 위해 LangChain에서 추가로 제공하는 것이 **LangGraph**이다.

# Cache

- 응답 결과를 저장해서 같은 질문이 들어오면 LLM에 요청하지 않고 저장된 결과를 보여주도록 한다.
    - 처리속도와 비용을 절감할 수 있다.
    - 특히 chatbot같이 비슷한 질문을 하는 경우 유용하다.
- 저장 방식은 `메모리`, `sqlite` 등 다양한 방식을 지원한다.
  
    ```python
    set_llm_cache(Cache객체)
    ```